# บทเรียน 05 - Agentic RAG


## Setup

สมุดบันทึกนี้สาธิตรูปแบบ Agentic RAG (Retrieval-Augmented Generation) โดยใช้ Microsoft Agent Framework

**ข้อกำหนดล่วงหน้า:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — จุดสิ้นสุดบริการ Azure AI Search ของคุณ
- `AZURE_SEARCH_API_KEY` — คีย์ API ของ Azure AI Search ของคุณ
- การปรับใช้ Azure OpenAI ที่กำหนดค่าผ่านตัวแปรแวดล้อม
- การรับรองความถูกต้อง Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG คืออะไร?

RAG แบบดั้งเดิมจะทำตามกระบวนการที่กำหนดไว้ล่วงหน้า: ดึงเอกสาร แล้วจึงสร้างคำตอบ **Agentic RAG** ก้าวไปอีกขั้นโดยให้ตัวแทนมีอิสระในการตัดสินใจ **เมื่อไหร่** และ **อย่างไร** ในการดึงข้อมูล

ด้วย Agentic RAG ตัวแทนสามารถ:
- **ตัดสินใจ** ว่าจำเป็นต้องดึงข้อมูลก่อนตอบคำถามหรือไม่
- **เลือก** แหล่งข้อมูลหรือเครื่องมือที่ต้องการสอบถาม
- **ประเมิน** ผลลัพธ์ที่ดึงมาและดำเนินการดึงข้อมูลซ้ำหากความพยายามครั้งแรกไม่เพียงพอ
- **รวม** ข้อมูลจากหลายขั้นตอนของการดึงข้อมูลเข้าด้วยกันเป็นคำตอบที่สมเหตุสมผล

สิ่งนี้ทำให้ตัวแทนมีความยืดหยุ่นและแม่นยำมากขึ้นเมื่อเทียบกับกระบวนการที่ตายตัวแบบดึงข้อมูลแล้วจึงสร้างคำตอบ


## การสร้างเครื่องมือค้นหา

ใน Agentic RAG แหล่งข้อมูลภายนอกจะถูกห่อหุ้มเป็น **เครื่องมือ** ที่ตัวแทนสามารถเรียกใช้ได้ตามต้องการ ซึ่งช่วยให้ตัวแทนสามารถถือว่าการค้นคืนข้อมูลเป็นเพียงอีกหนึ่งการกระทำที่สามารถทำได้ แทนที่จะเป็นขั้นตอนที่บังคับต้องทำ

ด้านล่างนี้เราจะกำหนดฐานความรู้ด้านการท่องเที่ยวและเปิดเผยเป็นเครื่องมือที่ตัวแทนสามารถเรียกใช้เพื่อค้นหาข้อมูลจุดหมายปลายทางได้


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## การสร้างตัวแทน RAG

ตอนนี้เราจะสร้างตัวแทนที่ได้รับคำสั่งให้ **ดึงข้อมูลก่อนตอบคำถามเสมอ** ตัวแทนจะใช้เครื่องมือ `search_travel_knowledge` เพื่อรองรับคำตอบของตัวเองจากฐานความรู้แทนที่จะพึ่งพาข้อมูลการฝึกของตนเอง


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## การค้นคืนแบบทำซ้ำ — รูปแบบ Maker-Checker

ข้อได้เปรียบที่สำคัญของ Agentic RAG คือ **การค้นคืนแบบทำซ้ำ** ตัวแทนสามารถทำการค้นหาหลายรอบเพื่อยืนยัน ปรับปรุง หรือขยายผลลัพธ์เริ่มต้น — คล้ายกับกระบวนการทำงาน "maker-checker":

1. **ขั้นตอน maker**: ตัวแทนดึงข้อมูลเบื้องต้นและร่างคำตอบ
2. **ขั้นตอน checker**: ตัวแทนทำการค้นคืนเพิ่มเติมเพื่อตรวจสอบรายละเอียดหรือเติมเต็มช่องว่าง

ด้านล่าง ตัวแทนถูกถามคำถามที่ต้องเปรียบเทียบจุดหมายปลายทางหลายแห่ง กระตุ้นให้ทำการค้นหาหลายครั้ง


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธีสร้างระบบ **Agentic RAG** โดยใช้ Microsoft Agent Framework:

- **Agentic RAG** ช่วยให้เอเจนต์ตัดสินใจอย่างอิสระเมื่อใดควรดึงข้อมูล ทำให้การดึงข้อมูลเป็นแบบไดนามิกไม่ใช่แบบคงที่
- **เครื่องมือเป็นแหล่งข้อมูล**: ฐานความรู้ภายนอก (เช่น Azure AI Search) ถูกห่อเป็นเครื่องมือที่เอเจนต์สามารถเรียกใช้ได้
- **การดึงข้อมูลแบบทำซ้ำ**: รูปแบบ maker-checker ช่วยให้เอเจนต์สามารถทำการดึงข้อมูลหลายรอบ — ค้นหา, ตรวจสอบ และปรับปรุง — ก่อนที่จะสร้างคำตอบสุดท้าย

ในสภาพแวดล้อมการใช้งานจริง คุณจะต้องเปลี่ยนจาก `TRAVEL_KNOWLEDGE_BASE` ที่อยู่ในหน่วยความจำ ไปเป็นดัชนีจริงของ Azure AI Search เพื่อรองรับการดึงเอกสารเกี่ยวกับการเดินทางในระดับใหญ่


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**:  
เอกสารฉบับนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) แม้เราจะพยายามให้ความถูกต้องสูงสุด แต่โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความคลาดเคลื่อนได้ เอกสารต้นฉบับในภาษาดั้งเดิมถือเป็นแหล่งข้อมูลที่ถูกต้องและน่าเชื่อถือที่สุด สำหรับข้อมูลที่สำคัญ ควรใช้การแปลโดยมืออาชีพซึ่งเป็นมนุษย์ เราจะไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความผิดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
